## LLM RAG Experimentation

woolworths_recipes**What is RAG?** Instead of fine-tuning an LLM on your data (expensive, slow), you:
1. Convert your data into vectors (numbers that capture meaning)
2. When a user asks a question, find the most relevant data using vector similarity
3. Feed that relevant data as context to the LLM, which generates an answer

**No LangChain.** Just numpy, Google's embedding API, and Gemini.

Install required packages (run once)
google-generativeai: Google's free API for embeddings + Gemini LLM
python-dotenv: loads API key from .env file
beautifulsoup4: used by the scraper (already installed likely)

In [1]:
!pip install google-generativeai python-dotenv numpy -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlepaddle 2.6.2 requires protobuf<=3.20.2,>=3.1.0; platform_system == "Windows", but you have protobuf 5.29.6 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Load API Key and test config

Google AI SDK and lists available embedding models to confirm the connection works.

In [1]:
import os
import json
import glob
import re
import numpy as np
from dotenv import load_dotenv

# Load the API key from the .env file at the workspace root
# Your .env has: GOOGLE_API_KEY = "AIzaSy..."
env_path = os.path.join("..", "..", ".env")
load_dotenv(env_path)

api_key = os.environ.get("GOOGLE_API_KEY", "").strip().strip('"')
print(f"API key loaded: {'Yes' if api_key else 'NO — check your .env file'}")
print(f"Key preview: {api_key[:10]}..." if api_key else "")

# Configure the Google AI SDK
import google.generativeai as genai
genai.configure(api_key=api_key)

print("\nAvailable embedding models:")
for m in genai.list_models():
    if "embed" in m.name:
        print(f"  {m.name}")

API key loaded: Yes
Key preview: AIzaSyDm1c...


d:\DEAKIN Post Grad Data Science\MASTERS UNITS\DiscountMate_Clone\DiscountMate_new\.venv\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
d:\DEAKIN Post Grad Data Science\MASTERS UNITS\DiscountMate_Clone\DiscountMate_new\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
d:\DEAKIN Post Grad Data Science\MASTERS UNITS\DiscountMate_Clone\DiscountMate_new\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets


Available embedding models:
  models/gemini-embedding-001
  models/gemini-embedding-2-preview


# inspect recipe structure (JSON)

For this test using a test folder of only 99 recipes. Actual deployment pipeline will contain over 13,000 recipes

In [2]:
# Point to the recipe directory (relative to this notebook)
RECIPE_DIR = os.path.join("..", "..", "woolworths_recipes")

# Load one recipe to inspect the raw JSON structure
json_files = sorted(glob.glob(os.path.join(RECIPE_DIR, "*.json")))
print(f"Found {len(json_files)} JSON files in {RECIPE_DIR}\n")

# Pick a recipe to examine
sample_path = [f for f in json_files if "chicken-and-mushroom" in f]
if not sample_path:
    sample_path = [json_files[0]]

with open(sample_path[0], "r", encoding="utf-8") as f:
    sample_recipe = json.load(f)

# Pretty print the raw JSON
print(f"FILE: {os.path.basename(sample_path[0])}")
print(f"TYPE: {sample_recipe.get('@type')}")
print(f"\nFull JSON structure:")
print(json.dumps(sample_recipe, indent=2, ensure_ascii=False)[:2000])

Found 99 JSON files in ..\..\woolworths_recipes

FILE: chicken-and-mushroom-green-curry.json
TYPE: Recipe

Full JSON structure:
{
  "@context": "https://schema.org",
  "@type": "Recipe",
  "@id": "https://www.woolworths.com.au/shop/recipes/chicken-and-mushroom-green-curry",
  "url": "https://www.woolworths.com.au/shop/recipes/chicken-and-mushroom-green-curry",
  "name": "Chicken & Mushroom Green Curry Recipe | Woolworths",
  "image": "https://foodhub.scene7.com/is/image/woolworthsltdprod/1108_chickenmushroomgreencurry:Square-1300x1300",
  "datePublished": "2026-04-09",
  "description": "Try our easy to follow Chicken & Mushroom Green Curry recipe. Absolutely delicious with the best ingredients from Woolworths.",
  "prepTime": "PT5M",
  "cookTime": "PT10M",
  "totalTime": "PT15M",
  "keywords": "Green curry,Eggplant,Gluten free,Tree nut free,Egg free,Low sugar,Sesame free,Mushroom,Quick & easy dinner,High protein,High fibre,Soy free,Wheat free,Chicken,Dairy free,Peanut free",
  "recipeI

## Preprocess: JSON into text chunk
This cell takes structured JSON and flattens it into a single rich text block. Run this and compare the output to the raw JSON from Cell 4, now all the semantic information is preserved but in a format an embedding model can digest.

In [4]:
def parse_duration(iso_duration):
    """Convert ISO 8601 duration (PT15M, PT1H30M) to readable string."""
    if not iso_duration:
        return ""
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?', iso_duration)
    if not match:
        return ""
    hours, minutes = match.groups()
    parts = []
    if hours:
        parts.append(f"{hours}h")
    if minutes:
        parts.append(f"{minutes}min")
    return " ".join(parts)


def recipe_json_to_text(recipe):
    """
    Convert a recipe JSON-LD dict into a single text chunk for embedding.
    
    WHY? Embedding models work on plain text, not JSON fields.
    We need to flatten all the useful information into one paragraph
    that captures the recipe's meaning — name, ingredients, cuisine, 
    cooking time, dietary tags, steps, nutrition.
    
    A user asking "quick chicken dinner" should match recipes that have:
    - "chicken" in ingredients
    - short cooking time  
    - "Mains" or "dinner" in category/keywords
    """
    parts = []

    # Recipe name (strip Woolworths branding)
    name = recipe.get("name", "").replace(" Recipe | Woolworths", "").strip()
    if name:
        parts.append(f"Recipe: {name}")

    # Description — skip the generic boilerplate every recipe has
    desc = recipe.get("description", "")
    if desc and "Absolutely delicious with the best ingredients" not in desc:
        parts.append(f"Description: {desc}")

    # Category (Mains, Dessert, Entrees) and Cuisine (Thai, Italian, etc.)
    category = recipe.get("recipeCategory", "")
    cuisine = recipe.get("recipeCuisine", "")
    if category or cuisine:
        meta = []
        if category: meta.append(f"Category: {category}")
        if cuisine: meta.append(f"Cuisine: {cuisine}")
        parts.append(" | ".join(meta))

    # Servings
    servings = recipe.get("recipeYield", "")
    if servings:
        parts.append(f"Serves: {servings}")

    # Timing
    prep = parse_duration(recipe.get("prepTime"))
    cook = parse_duration(recipe.get("cookTime"))
    total = parse_duration(recipe.get("totalTime"))
    time_parts = []
    if prep: time_parts.append(f"Prep: {prep}")
    if cook: time_parts.append(f"Cook: {cook}")
    if total: time_parts.append(f"Total: {total}")
    if time_parts:
        parts.append(" | ".join(time_parts))

    # Keywords — dietary tags are gold for matching
    keywords = recipe.get("keywords", "")
    if keywords:
        parts.append(f"Tags: {keywords}")

    # Ingredients — the most important field for "what can I make with X?" queries
    ingredients = recipe.get("recipeIngredient", [])
    if ingredients:
        parts.append("Ingredients: " + "; ".join(ingredients))

    # Instructions
    instructions = recipe.get("recipeInstructions", [])
    if instructions:
        steps = []
        for i, step in enumerate(instructions, 1):
            text = step.get("text", "") if isinstance(step, dict) else str(step)
            if text:
                steps.append(f"Step {i}: {text}")
        if steps:
            parts.append("Instructions:\n" + "\n".join(steps))

    # Nutrition
    nutrition = recipe.get("nutrition", {})
    if nutrition and isinstance(nutrition, dict):
        nutri_parts = []
        for key in ["calories", "proteinContent", "fatContent",
                     "carbohydrateContent", "fiberContent", "sugarContent"]:
            val = nutrition.get(key, "")
            if val:
                label = key.replace("Content", "").capitalize()
                nutri_parts.append(f"{label}: {val}")
        if nutri_parts:
            parts.append("Nutrition: " + " | ".join(nutri_parts))

    return "\n".join(parts)


# DEMO: Show the transformation
text_chunk = recipe_json_to_text(sample_recipe)

print("=" * 70)
print("BEFORE (raw JSON) → AFTER (text chunk for embedding)")
print("=" * 70)
print(text_chunk)
print("=" * 70)
print(f"\nChunk length: {len(text_chunk)} characters, ~{len(text_chunk.split())} words")

BEFORE (raw JSON) → AFTER (text chunk for embedding)
Recipe: Chicken & Mushroom Green Curry
Category: Mains | Cuisine: Thai
Serves: 4
Prep: 5min | Cook: 10min | Total: 15min
Tags: Green curry,Eggplant,Gluten free,Tree nut free,Egg free,Low sugar,Sesame free,Mushroom,Quick & easy dinner,High protein,High fibre,Soy free,Wheat free,Chicken,Dairy free,Peanut free
Ingredients: 1 tsp sugar; 3 tbs curry paste; 1 tsp squid brand fish sauce; 400g chicken fillet, cut into-bite sized pieces; 1 cup fresh basil leaves; 1 cup eggplant, cubed; 2 cup button mushrooms; 0.5 jar kaffir lime leaves, thinly sliced; 400ml coconut milk
Instructions:
Step 1: Bring coconut milk, green curry paste, fish sauce and sugar to the boil in a wok, stirring occasionally.
Step 2: Add chicken, mushroom and eggplant to wok. Simmer for 2 minutes or until chicken is cooked.
Step 3: Add kaffir lime leaves and basil leaves. Simmer for 1 minute and remove from heat. Serve with steamed rice
Nutrition: Calories: 335 calories | P

## Load all recipes

Processes every recipe JSON through the preprocessor. You should see ~100 recipes loaded with their names and cuisines.

In [5]:
def recipe_json_to_metadata(recipe):
    """Extract structured metadata for display alongside results."""
    name = recipe.get("name", "").replace(" Recipe | Woolworths", "").strip()
    return {
        "name": name,
        "slug": recipe.get("recipeNameId", ""),
        "url": recipe.get("url", ""),
        "category": recipe.get("recipeCategory", ""),
        "cuisine": recipe.get("recipeCuisine", ""),
        "keywords": recipe.get("keywords", ""),
        "total_time": parse_duration(recipe.get("totalTime")),
        "ingredients": recipe.get("recipeIngredient", []),
        "num_ingredients": len(recipe.get("recipeIngredient", [])),
    }


# Load all recipe JSON files
all_recipes = []
for filepath in sorted(glob.glob(os.path.join(RECIPE_DIR, "*.json"))):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            raw = json.load(f)
        if raw.get("@type") != "Recipe":
            continue
        all_recipes.append({
            "text": recipe_json_to_text(raw),
            "metadata": recipe_json_to_metadata(raw),
            "source_file": os.path.basename(filepath),
        })
    except (json.JSONDecodeError, KeyError) as e:
        print(f"  Skip: {os.path.basename(filepath)} — {e}")

print(f"Loaded {len(all_recipes)} recipes")
print(f"Avg text chunk: {sum(len(r['text']) for r in all_recipes) // len(all_recipes)} chars")
print(f"\nSample recipe names:")
for r in all_recipes[:5]:
    print(f"  • {r['metadata']['name']} ({r['metadata']['cuisine']}, {r['metadata']['total_time']})")

Loaded 99 recipes
Avg text chunk: 1251 chars

Sample recipe names:
  • Apple & Rhubarb Pie (British, 90min)
  • Apple & Walnut Coffee Cake (North American, 60min)
  • Apple Crumble (British, 70min)
  • Asian Salad with Avocado & Salmon (Asian, 10min)
  • Aussie Beef Burger (Australian, 15min)


## Embed ONE recipe (see what a vector looks like)

In [6]:
# Embed a single recipe text to see what the output looks like
sample_text = all_recipes[0]["text"]

result = genai.embed_content(
    model="models/gemini-embedding-001",
    content=sample_text,
    task_type="retrieval_document",  # tells the model this is a document to be searched
)

embedding = np.array(result["embedding"])

print(f"Input text (first 200 chars): {sample_text[:200]}...")
print(f"\nEmbedding shape: {embedding.shape}")
print(f"Embedding dtype: {embedding.dtype}")
print(f"First 10 values: {embedding[:10]}")
print(f"Value range: [{embedding.min():.4f}, {embedding.max():.4f}]")
print(f"\nThis vector of {len(embedding)} numbers IS the recipe's meaning.")
print("Similar recipes will have similar vectors.")

Input text (first 200 chars): Recipe: Apple & Rhubarb Pie
Category: Desserts | Cuisine: British
Serves: 6
Prep: 50min | Cook: 40min | Total: 90min
Tags: Apple pie & tart,Seafood free,Soy free,Apple,Sesame free,Rhubarb
Ingredients:...

Embedding shape: (3072,)
Embedding dtype: float64
First 10 values: [ 1.8061951e-03  1.1613825e-02  2.2897908e-02 -7.0352330e-02
 -1.0129425e-02 -1.5483750e-02 -8.4272330e-05 -4.8187390e-03
  8.7388740e-03  4.2704810e-03]
Value range: [-0.2078, 0.2222]

This vector of 3072 numbers IS the recipe's meaning.
Similar recipes will have similar vectors.


# Batch Emebed all 100 test recipes

this builds the index

 Sends all recipe texts to Google's embedding API. For ~100 recipes this is 1 batch call and takes a few seconds. The result is a matrix where row i is the vector for recipe i. This is your "vector store" no database needed for now. 

In [7]:
import time

# Extract all text chunks
texts = [r["text"] for r in all_recipes]

# Smaller batches with delay to stay within free tier rate limits
BATCH_SIZE = 10  # smaller batches to avoid rate limits
DELAY = 15        # seconds between batches

print(f"Embedding {len(texts)} recipes in batches of {BATCH_SIZE}...")
print(f"(With {DELAY}s delay between batches to respect free tier limits)\n")

all_embeddings = []
start_time = time.time()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    
    # Retry logic in case we still hit the limit
    for attempt in range(3):
        try:
            result = genai.embed_content(
                model="models/gemini-embedding-001",
                content=batch,
                task_type="retrieval_document",
            )
            all_embeddings.extend(result["embedding"])
            print(f"  Batch {i//BATCH_SIZE + 1}: embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)} recipes")
            break
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                wait = DELAY * (attempt + 2)
                print(f"  Rate limited — waiting {wait}s before retry {attempt + 1}/3...")
                time.sleep(wait)
            else:
                raise
    
    # Delay between batches
    if i + BATCH_SIZE < len(texts):
        time.sleep(DELAY)

elapsed = time.time() - start_time
embeddings = np.array(all_embeddings, dtype=np.float32)

print(f"\nDone in {elapsed:.1f}s")
print(f"Embeddings matrix shape: {embeddings.shape}")
print(f"  → {embeddings.shape[0]} recipes × {embeddings.shape[1]} dimensions")
print(f"  → Memory: {embeddings.nbytes / 1024:.1f} KB")
print(f"\nThis is your entire 'vector database' — just a numpy array.")

Embedding 99 recipes in batches of 10...
(With 15s delay between batches to respect free tier limits)

  Batch 1: embedded 10/99 recipes
  Batch 2: embedded 20/99 recipes
  Batch 3: embedded 30/99 recipes
  Batch 4: embedded 40/99 recipes
  Batch 5: embedded 50/99 recipes
  Batch 6: embedded 60/99 recipes
  Batch 7: embedded 70/99 recipes
  Batch 8: embedded 80/99 recipes
  Batch 9: embedded 90/99 recipes
  Batch 10: embedded 99/99 recipes

Done in 154.9s
Embeddings matrix shape: (99, 3072)
  → 99 recipes × 3072 dimensions
  → Memory: 1188.0 KB

This is your entire 'vector database' — just a numpy array.


# Find recipes by meaning  -- "R" (AG) [Retrieval Step]
This is retrieval step of RAG, try different query strings

In [8]:
def search_recipes(query, top_k=5):
    """
    Semantic search: embed the query, then find the most similar recipe vectors.
    
    Cosine similarity = how much two vectors point in the same direction.
    Score of 1.0 = identical meaning, 0.0 = completely unrelated.
    """
    # Embed the query (note: task_type is "retrieval_query" not "retrieval_document")
    result = genai.embed_content(
        model="models/gemini-embedding-001",
        content=query,
        task_type="retrieval_query",
    )
    query_vec = np.array(result["embedding"], dtype=np.float32)
    
    # Cosine similarity against all recipe vectors
    query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-10)
    emb_norms = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10)
    scores = emb_norms @ query_norm
    
    # Get top-k indices
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "metadata": all_recipes[idx]["metadata"],
            "text": all_recipes[idx]["text"],
            "score": float(scores[idx]),
        })
    return results


# TEST: Try a search
query = "quick chicken dinner"
print(f"QUERY: \"{query}\"\n")
print("=" * 70)

results = search_recipes(query, top_k=5)

for i, r in enumerate(results, 1):
    m = r["metadata"]
    print(f"\n#{i} — {m['name']}  (score: {r['score']:.3f})")
    print(f"   Cuisine: {m['cuisine']} | Time: {m['total_time']} | Ingredients: {m['num_ingredients']}")
    print(f"   Top ingredients: {', '.join(m['ingredients'][:4])}")

QUERY: "quick chicken dinner"


#1 — Chargrilled Chicken with Fresh Pineapple & Red Onion  (score: 0.710)
   Cuisine: Australian | Time: 25min | Ingredients: 6
   Top ingredients: 1 small pineapple, peeled and sliced, 1 large red onion, cut into thin wedges, 4 chicken breast fillets, salad greens (to serve)

#2 — Chicken & Mushroom Green Curry  (score: 0.696)
   Cuisine: Thai | Time: 15min | Ingredients: 9
   Top ingredients: 1 tsp sugar, 3 tbs curry paste, 1 tsp squid brand fish sauce, 400g chicken fillet, cut into-bite sized pieces

#3 — Moroccan Chicken with Couscous Salad  (score: 0.680)
   Cuisine: North African | Time: 25min | Ingredients: 15
   Top ingredients: 1 clove garlic, finely chopped, 2 tbs coriander, leaves, 2 tbs mint, chopped, 1 small red onion, finely chopped

#4 — Chicken Pasta Soup with Mozzarella Toast  (score: 0.680)
   Cuisine: Italian | Time: 40min | Ingredients: 9
   Top ingredients: 700g chunky pasta sauce, 2 tbs basil (small leaves to serve), 1 cooked fresh 

## FULL RAG Pipeline

This is the complete pipeline. It retrieves recipes, builds a prompt with them as context, and sends it to Gemini. The LLM answers only from your recipe data — it can't hallucinate recipes that don't exist in your collection.

Due to getting rate limits on even one query, first trying to see if there are any other models available via gemini/google before moving to open router and/or free hugging face models which could perform the the task to an equal degree based on past experience in the deep learning subject. 

In [16]:
for m in genai.list_models():
    if "generate" in str(m.supported_generation_methods).lower():
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [17]:
SYSTEM_PROMPT = """You are DiscountMate's Recipe Assistant. You help users find recipes 
and meal ideas based on the Woolworths recipe collection.

Rules:
- Only recommend recipes from the provided context. Do not invent recipes.
- If the context doesn't contain a good match, say so honestly.
- Include the recipe name, key ingredients, and a brief summary.
- Keep responses concise — 2-4 short paragraphs max.
"""

def rag_query(user_question, top_k=5):
    """
    Full RAG pipeline:
    1. Retrieve relevant recipes via semantic search
    2. Build a prompt with those recipes as context
    3. Send to Gemini to generate a natural-language answer
    """
    # Step 1: Retrieve
    results = search_recipes(user_question, top_k=top_k)
    
    # Step 2: Build context
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(f"--- Recipe {i} (relevance: {r['score']:.2f}) ---\n{r['text']}\n")
    context = "\n".join(context_parts)
    
    prompt = (
        f"Based on the following recipes from our database, "
        f"answer the user's question.\n\n"
        f"RECIPE CONTEXT:\n{context}\n\n"
        f"USER QUESTION: {user_question}"
    )
    
    # Step 3: Generate with Gemini
    #model = genai.GenerativeModel("gemini-2.0-flash", system_instruction=SYSTEM_PROMPT) Hitting rate limits here, so using a smaller model for testing
    #model = genai.GenerativeModel("gemini-2.0-flash-lite", system_instruction=SYSTEM_PROMPT)
    #model = genai.GenerativeModel("gemini-1.5-flash", system_instruction=SYSTEM_PROMPT)
    model = genai.GenerativeModel("gemini-2.5-flash-lite", system_instruction=SYSTEM_PROMPT) # this one worked on first attempt
    response = model.generate_content(prompt)
    
    return {
        "answer": response.text,
        "sources": [r["metadata"]["name"] for r in results],
        "scores": [r["score"] for r in results],
    }


# TEST IT
question = "I feel like falafel tonight, what do you suggest?"
print(f"USER: {question}\n")
print("=" * 70)

result = rag_query(question)

print(f"\nMATEBOT:\n{result['answer']}")
print(f"\n📖 Sources (with relevance scores):")
for name, score in zip(result["sources"], result["scores"]):
    print(f"   {score:.3f} — {name}")

USER: I feel like falafel tonight, what do you suggest?


MATEBOT:
For a falafel dish, I suggest the **Chickpea Falafel with Spicy Hummus**.

This recipe serves 4 and takes about 55 minutes in total. Key ingredients include chickpeas, plain flour, egg, onion, garlic, coriander, cumin, parsley, and pita bread.

The falafel are made by processing chickpeas with spices and herbs, then binding with flour and egg. The mixture is shaped into patties and fried until golden. They are served on toasted pita bread with sliced tomato, chopped coriander, and hummus.

📖 Sources (with relevance scores):
   0.722 — Chickpea Falafel with Spicy Hummus
   0.635 — Turkish Pizza with Avocado Tzatziki
   0.632 — Lentil Patties with Beetroot, Spinach & Feta Salad
   0.629 — Grilled Lamb with Buckwheat & Pine-Nut Tabouli
   0.624 — Vegetable Rolls


# Test 2 Changing the system prompt 

In [19]:
SYSTEM_PROMPT = """You are DiscountMate's Recipe Assistant. You help users find recipes 
and meal ideas based on the Woolworths recipe collection.

Rules:
- Only recommend recipes from the provided context. Do not invent recipes.
- If the context doesn't contain a good match, say so honestly.
- When the user asks for multiple recipes (e.g. "5 meals for the week"), list each one 
  with its name, cuisine, total time, and key ingredients.
- When the user asks for a specific recipe or says "show me the recipe", include:
  the full ingredient list, step-by-step instructions, and nutrition info if available.
- For general questions, include the recipe name, key ingredients, and a brief summary.
- Format responses clearly with headings and bullet points for readability.
"""

def rag_query(user_question, top_k=5):
    """
    Full RAG pipeline:
    1. Retrieve relevant recipes via semantic search
    2. Build a prompt with those recipes as context
    3. Send to Gemini to generate a natural-language answer
    
    top_k controls how many recipes are retrieved as context.
    Increase it for broader questions ("5 meals for the week").
    """
    # Step 1: Retrieve
    results = search_recipes(user_question, top_k=top_k)
    
    # Step 2: Build context
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(f"--- Recipe {i} (relevance: {r['score']:.2f}) ---\n{r['text']}\n")
    context = "\n".join(context_parts)
    
    prompt = (
        f"Based on the following recipes from our database, "
        f"answer the user's question.\n\n"
        f"RECIPE CONTEXT:\n{context}\n\n"
        f"USER QUESTION: {user_question}"
    )
    
    # Step 3: Generate with Gemini
    model = genai.GenerativeModel("gemini-2.5-flash-lite", system_instruction=SYSTEM_PROMPT)
    response = model.generate_content(prompt)
    
    return {
        "answer": response.text,
        "sources": [r["metadata"]["name"] for r in results],
        "scores": [r["score"] for r in results],
    }


# TEST 1: Single recipe request (detailed)
question = "Show me the full recipe for the chickpea falafel"
print(f"USER: {question}\n")
print("=" * 70)
result = rag_query(question)
print(f"\nDSCM8-GPT:\n{result['answer']}")
print(f"\n📖 Sources: {', '.join(result['sources'])}")

print("\n\n" + "=" * 70)

# TEST 2: Multi-recipe request for the week
question2 = "Suggest 5 different easy dinners for the week, variety of cuisines please"
print(f"\nUSER: {question2}\n")
print("=" * 70)
result2 = rag_query(question2, top_k=8)  # retrieve more so Gemini has options
print(f"\nDSCM8-GPT:\n{result2['answer']}")
print(f"\n📖 Sources: {', '.join(result2['sources'])}")

USER: Show me the full recipe for the chickpea falafel


DSCM8-GPT:
Here is the full recipe for Chickpea Falafel with Spicy Hummus:

**Recipe: Chickpea Falafel with Spicy Hummus**
Cuisine: Middle Eastern
Total Time: 55min

**Ingredients:**

*   3 tbs plain flour
*   2 tomato, sliced (to serve)
*   2 tbs coriander, chopped (to serve)
*   0.5 small red onion
*   0.25 cup parsley, chopped
*   2 tsp ground coriander
*   2 tsp ground cumin
*   800g canned chickpeas no added salt, drained
*   0.5 cup hommus dip (to serve)
*   1 egg, beaten
*   3 garlic cloves, crushed
*   1L vegetable oil (for frying)
*   4 small pita bread

**Instructions:**

**Step 1:** Process chickpeas, garlic, onion, coriander, cumin and parsley in a food processor until just smooth. Transfer to a bowl and add flour and egg. Shape mixture into 8 patties and refrigerate for 30 minutes to firm.
**Step 2:** Heat oil in a large frying pan or heat a BBQ hotplate on medium-high. Cook falafel patties for 5 minutes each side or

# Test 3 - full recipe configuration

In [21]:
SYSTEM_PROMPT = """You are DiscountMate's Recipe Assistant. You help users find recipes 
and meal ideas based on the Woolworths recipe collection.

Rules:
- Only recommend recipes from the provided context. Do not invent recipes.
- If the context doesn't contain a good match, say so honestly.
- When the user asks for multiple recipes (e.g. "5 meals for the week"), list each one 
  with its name, cuisine, total time, ALL ingredients (not just key ones), 
  and summarised step-by-step instructions.
- When the user asks for a specific recipe or says "show me the recipe", include:
  the full ingredient list, step-by-step instructions, and nutrition info if available.
- For general questions, include the recipe name, key ingredients, and a brief summary.
- Always list ALL ingredients, never abbreviate to just "key ingredients".
- Format responses clearly with headings and bullet points for readability.
"""

def rag_query(user_question, top_k=5):
    """
    Full RAG pipeline:
    1. Retrieve relevant recipes via semantic search
    2. Build a prompt with those recipes as context
    3. Send to Gemini to generate a natural-language answer
    
    top_k controls how many recipes are retrieved as context.
    Increase it for broader questions ("5 meals for the week").
    """
    # Step 1: Retrieve
    results = search_recipes(user_question, top_k=top_k)
    
    # Step 2: Build context
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(f"--- Recipe {i} (relevance: {r['score']:.2f}) ---\n{r['text']}\n")
    context = "\n".join(context_parts)
    
    prompt = (
        f"Based on the following recipes from our database, "
        f"answer the user's question.\n\n"
        f"RECIPE CONTEXT:\n{context}\n\n"
        f"USER QUESTION: {user_question}"
    )
    
    # Step 3: Generate with Gemini
    model = genai.GenerativeModel("gemini-2.5-flash-lite", system_instruction=SYSTEM_PROMPT)
    response = model.generate_content(prompt)
    
    return {
        "answer": response.text,
        "sources": [r["metadata"]["name"] for r in results],
        "scores": [r["score"] for r in results],
    }


# TEST: Multi-recipe with full details
question = "Suggest 5 different easy dinners for the week with full recipes and all ingredients"
print(f"USER: {question}\n")
print("=" * 70)
result = rag_query(question, top_k=8)
print(f"\nDSCM8-GPT:\n{result['answer']}")
print(f"\n📖 Sources: {', '.join(result['sources'])}")

USER: Suggest 5 different easy dinners for the week with full recipes and all ingredients


DSCM8-GPT:
Here are 5 easy dinner recipes for your week:

---

### 1. Sausage & Lentil Casserole

*   **Cuisine:** Australian
*   **Total Time:** 25min

**Ingredients:**
*   400g broccoli, steamed (to serve)
*   1 brown onion, thinly sliced
*   0.33 cup parsley, roughly chopped
*   400g can diced tomatoes
*   400g can lentils, drained, rinsed
*   1 tbs olive oil
*   500g extra-lean pork sausages

**Instructions:**
1.  Heat oil in a non-stick frying pan over a medium heat. Add onion and cook for 5 minutes or until tender. Transfer onion to a bowl, set aside.
2.  Squeeze meatball-size portions from sausages into the frying pan. Cook for 8 minutes, shaking pan often, or until meatballs are browned on all sides.
3.  Add onion, tomatoes, lentils and water to the pan. Stir until mixture comes to the boil.
4.  Cover pan and cook for 2 minutes. Spoon into shallow bowls. Garnish with parsley. Serve with 

https://huggingface.co/spaces/mteb/leaderboard?ref=codesphere.ghost.io

Best sentence embedding models 
If you are looking for the best open-source sentence embedding models, the HuggingFace mteb leaderboard is where you should look. With the highest overall performance on multiple benchmarks, bge-en-icl stands out as the best model for sentence embeddings. With 7.11B parameters, it has the highest overall average score and performs well across most categories, which makes it highly reliable and versatile. However, if memory usage and modal size are an issue, then stella_en_1.5B_v5 is a great lightweight alternative in comparison to the first one. This model has significantly lower memory requirements and closely matches the performance of bge-en-icl, even surpassing it on some tasks like STS and summarization.

https://codesphere.com/articles/best-open-source-sentence-embedding-models#:~:text=Sometimes%20your%20use%20case%20just,to%20download%20the%20required%20parameters.

---
# V2: Local Sentence-Transformers + OpenRouter RAG

**Why V2?** Google's free API has harsh rate limits that make 13k recipe processing impossible. 
This version uses:
- **sentence-transformers** (local, free, unlimited) for ALL embeddings
- **OpenRouter** (free tier Gemini access via separate quota) for LLM generation
- **Ingredient extraction** — embed ingredients separately for future product/SKU matching

**Key insight**: The embedding model and the generation model are completely independent.
Sentence-transformers converts text→vectors for search. OpenRouter's LLM reads the search 
results as plain text and writes an answer. They never interact directly.

## installs

In [22]:
# Cell V2-1: Install sentence-transformers + requests (for OpenRouter)
# sentence-transformers runs 100% locally — no API keys, no rate limits
!pip install sentence-transformers requests --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Load Sentence Transformer models

Load all-mpnet-base-v2 model (~420MB first run)

## Model switch 

Edit: Note I had to actually switch back to the all-mpnet-base-v2 with a shorter token window, the long context of the Kina model was casuing errors at inference time: The real problem is jina diluting semantic signal on long texts. Your recipe embeddings include full step-by-step instructions, which are long but semantically noise for search. "Preheat oven to 180°C" doesn't help match "chicken dinner". Jina spreads the embedding across all that text, washing out the "chicken" signal.

**I've switched from all-mp-net-base-v2 to jina-embeddings-v2-base-en since the earlier modle only had max embedding tokens of 284 and for full ingredient matching later we don't want silent truncation. The new model has much larger token and still should perform the task adequetly**

Switched to sentence transformer for embedding locally, since gemini embedding tool is rate limited in the above version. This does not effect the model used the cosine retrieval will embed the query and lookup the recipe via cosine, and this is returned to the LM in natural language for it to use via its normal processes. 

In [3]:
# Cell V2-2: Load sentence-transformers model
# all-mpnet-base-v2 — best quality/size tradeoff on MTEB leaderboard
# ~420MB download on first run, then cached locally
# Produces 768-dimensional embeddings (same as gemini-embedding-001)

from sentence_transformers import SentenceTransformer
import numpy as np
import time

model = SentenceTransformer('all-mpnet-base-v2')
#model = SentenceTransformer('jinaai/jina-embeddings-v2-base-en') #this models context and embedding space was too sparse it lost context
print(f"Model loaded: all-mpnet-base-v2")
print(f"Max sequence length: {model.max_seq_length}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 264.39it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: jina-embeddings-v2-base-en
Max sequence length: 384
Embedding dimension: 768


### Load & preprocess all ~100 recipes (reuses v1 parse logic)

In [4]:
# Cell V2-3: Load and preprocess ALL recipes (reuse v1 logic)
import json, os, re

RECIPE_DIR = "../../woolworths_recipes"

def parse_duration(iso_str):
    """Convert ISO 8601 duration (PT30M, PT1H15M) to readable string."""
    if not iso_str:
        return "Not specified"
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?', iso_str)
    if not match:
        return iso_str
    hours, minutes = match.groups()
    parts = []
    if hours: parts.append(f"{hours}h")
    if minutes: parts.append(f"{minutes}m")
    return " ".join(parts) if parts else "Not specified"

def recipe_json_to_text(data):
    """Convert recipe JSON-LD to a single text chunk for embedding."""
    recipe = None
    if isinstance(data, list):
        for item in data:
            if item.get("@type") == "Recipe":
                recipe = item
                break
    elif isinstance(data, dict) and data.get("@type") == "Recipe":
        recipe = data
    if not recipe:
        return None, None

    name = recipe.get("name", "Unknown")
    # Skip generic Woolworths boilerplate
    if name.lower().startswith("woolworths"):
        return None, None

    description = recipe.get("description", "")
    ingredients = recipe.get("recipeIngredient", [])
    instructions = recipe.get("recipeInstructions", [])
    prep_time = parse_duration(recipe.get("prepTime"))
    cook_time = parse_duration(recipe.get("cookTime"))
    servings = recipe.get("recipeYield", "Not specified")
    category = recipe.get("recipeCategory", "Not specified")
    cuisine = recipe.get("recipeCuisine", "Not specified")

    steps = []
    for inst in instructions:
        if isinstance(inst, dict):
            steps.append(inst.get("text", ""))
        elif isinstance(inst, str):
            steps.append(inst)

    text = f"""Recipe: {name}
Description: {description}
Category: {category} | Cuisine: {cuisine}
Prep: {prep_time} | Cook: {cook_time} | Serves: {servings}

Ingredients:
{chr(10).join(f'- {i}' for i in ingredients)}

Instructions:
{chr(10).join(f'{j+1}. {s}' for j, s in enumerate(steps))}"""

    metadata = {
        "name": name,
        "description": description,
        "ingredients": ingredients,
        "category": category,
        "cuisine": cuisine,
        "prep_time": prep_time,
        "cook_time": cook_time,
        "servings": servings,
        "instructions": steps
    }
    return text, metadata

# Load all recipes
recipe_texts = []
recipe_metadata = []
json_files = sorted([f for f in os.listdir(RECIPE_DIR) if f.endswith('.json')])

for fname in json_files:
    with open(os.path.join(RECIPE_DIR, fname), 'r', encoding='utf-8') as f:
        data = json.load(f)
    text, meta = recipe_json_to_text(data)
    if text and meta:
        meta["source_file"] = fname
        recipe_texts.append(text)
        recipe_metadata.append(meta)

print(f"Loaded {len(recipe_texts)} recipes from {len(json_files)} JSON files")
print(f"Example recipe: {recipe_metadata[0]['name']}")

Loaded 99 recipes from 99 JSON files
Example recipe: Apple & Rhubarb Pie Recipe | Woolworths


### 	Embed ALL recipes locally — no rate limits, batch_size=32

In [5]:
# Cell V2-4: Embed ALL recipes locally with sentence-transformers
# No API calls, no rate limits — runs on your CPU/GPU
# ~100 recipes takes seconds, 13k recipes would take a few minutes

start = time.time()
recipe_embeddings = model.encode(recipe_texts, show_progress_bar=True, batch_size=32)
elapsed = time.time() - start

print(f"\nEmbedded {len(recipe_embeddings)} recipes in {elapsed:.1f}s")
print(f"Embedding shape: {recipe_embeddings.shape}")
print(f"Speed: {len(recipe_embeddings)/elapsed:.1f} recipes/sec")
print(f"At this speed, 13k recipes would take ~{13000/len(recipe_embeddings)*elapsed:.0f}s")

Batches: 100%|██████████| 4/4 [01:37<00:00, 24.26s/it]


Embedded 99 recipes in 97.9s
Embedding shape: (99, 768)
Speed: 1.0 recipes/sec
At this speed, 13k recipes would take ~12852s


## Ingredient Extraction & Embedding

Extract each recipe's raw ingredient strings and embed them independently.
This creates a separate vector store so we can later match **on-sale products → ingredients → recipes**.

**Why raw strings?** Ingredient text like "2 cups self-raising flour" is messy, but cosine similarity 
handles fuzzy matching well. A product like "Woolworths Self Raising Flour 1kg" will still score 
highly against "2 cups self-raising flour" because the embedding captures semantic meaning, 
not exact string matching.

In [6]:
# Cell V2-5: Extract all unique ingredients and build ingredient→recipe mapping

ingredient_to_recipes = {}  # ingredient_text → list of recipe indices
all_ingredients = []        # flat list of unique ingredient strings

for idx, meta in enumerate(recipe_metadata):
    for ing in meta["ingredients"]:
        ing_clean = ing.strip()
        if ing_clean not in ingredient_to_recipes:
            ingredient_to_recipes[ing_clean] = []
            all_ingredients.append(ing_clean)
        ingredient_to_recipes[ing_clean].append(idx)

print(f"Total unique ingredients: {len(all_ingredients)}")
print(f"Total recipes: {len(recipe_metadata)}")
print(f"Avg ingredients per recipe: {sum(len(m['ingredients']) for m in recipe_metadata)/len(recipe_metadata):.1f}")
print(f"\nSample ingredients:")
for ing in all_ingredients[:10]:
    recipes_using = [recipe_metadata[i]['name'] for i in ingredient_to_recipes[ing]]
    print(f"  '{ing}' → used in {len(recipes_using)} recipe(s): {recipes_using[0]}")

Total unique ingredients: 779
Total recipes: 99
Avg ingredients per recipe: 9.4

Sample ingredients:
  '1.5 cup plain flour' → used in 2 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '1 tbs cornflour' → used in 2 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '125g butter, chopped' → used in 2 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '2 tbs eggs' → used in 1 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '1 bunch rhubarb, trimed, cut into 2 cm pieces (to serve)' → used in 1 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '1 tbs white sugar' → used in 1 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '2 tbs almond meal' → used in 1 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '2 granny smith apples, peeled, cored, thinly sliced' → used in 1 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '6 scoop vanilla ice cream (to serve)' → used in 1 recipe(s): Apple & Rhubarb Pie Recipe | Woolworths
  '3 tsp instant coffee powder' → used in 1 re

Embed all ingredients + sanity check ("chicken breast" search)

In [7]:
# Cell V2-6: Embed all unique ingredients
# Same model = same embedding space — ingredients and recipes are comparable

start = time.time()
ingredient_embeddings = model.encode(all_ingredients, show_progress_bar=True, batch_size=64)
elapsed = time.time() - start

print(f"\nEmbedded {len(ingredient_embeddings)} unique ingredients in {elapsed:.1f}s")
print(f"Embedding shape: {ingredient_embeddings.shape}")

# Quick sanity check: find ingredients most similar to "chicken"
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

test_query = model.encode(["chicken breast"])
sims = [cosine_sim(test_query[0], emb) for emb in ingredient_embeddings]
top_5 = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)[:5]

print(f"\nSanity check — ingredients most similar to 'chicken breast':")
for idx, score in top_5:
    print(f"  {score:.3f} | {all_ingredients[idx]}")

Batches: 100%|██████████| 13/13 [00:02<00:00,  4.93it/s]



Embedded 779 unique ingredients in 2.7s
Embedding shape: (779, 768)

Sanity check — ingredients most similar to 'chicken breast':
  0.684 | 2 chicken breast fillets
  0.675 | 4 chicken breast fillets
  0.666 | 1 whole chicken
  0.618 | 1 cooked fresh chicken whole
  0.568 | 6 chicken thigh cutlets


### Local semantic search function (cosine similarity)

In [8]:
# Cell V2-7: Semantic search function (local, no API calls)

def search_recipes_v2(query, top_k=5):
    """Search recipes using sentence-transformers embeddings with cosine similarity."""
    query_emb = model.encode([query])
    
    # Proper cosine similarity — normalize both vectors
    query_norm = query_emb / (np.linalg.norm(query_emb, axis=1, keepdims=True) + 1e-10)
    emb_norms = recipe_embeddings / (np.linalg.norm(recipe_embeddings, axis=1, keepdims=True) + 1e-10)
    sims = np.dot(emb_norms, query_norm.T).flatten()
    
    top_indices = np.argsort(sims)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "rank": len(results) + 1,
            "score": float(sims[idx]),
            "name": recipe_metadata[idx]["name"],
            "text": recipe_texts[idx],
            "metadata": recipe_metadata[idx]
        })
    return results

# Test it
results = search_recipes_v2("quick chicken dinner for family")
print("Search: 'quick chicken dinner for family'\n")
for r in results:
    print(f"  #{r['rank']} ({r['score']:.3f}) {r['name']}")
    print(f"    Category: {r['metadata']['category']} | Cook: {r['metadata']['cook_time']}")
    print()

Search: 'quick chicken dinner for family'

  #1 (0.554) Chicken Pasta Soup with Mozzarella Toast Recipe | Woolworths
    Category: Entrees | Cook: 30m

  #2 (0.545) Roast Chicken & Vegetables Recipe | Woolworths
    Category: Mains | Cook: 70m

  #3 (0.534) Chicken, Leek & Mushroom Casserole Recipe | Woolworths
    Category: Mains | Cook: 120m

  #4 (0.529) Turkey Casserole Recipe | Woolworths
    Category: Mains | Cook: 35m

  #5 (0.491) Chargrilled Chicken with Fresh Pineapple & Red Onion Recipe | Woolworths
    Category: Mains | Cook: 10m



## OpenRouter LLM Generation

OpenRouter provides a unified API gateway to many LLMs with an OpenAI-compatible format.
The free tier includes `google/gemini-2.0-flash-exp:free` — separate quota from Google AI Studio.

**API format**: Same as OpenAI chat completions → easy to swap models later.

In [9]:
# Cell V2-8: OpenRouter generation setup
import requests
from dotenv import load_dotenv

load_dotenv("../../.env", override=True)
OPENROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
print(f"OpenRouter key loaded: {'Yes' if OPENROUTER_API_KEY else 'NO — check .env!'}")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

def generate_with_openrouter(system_prompt, user_message, temperature=0.7):
    """Call OpenRouter API (OpenAI-compatible format)."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://discountmate.app",
        "X-Title": "DiscountMate RAG"
    }
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        "temperature": temperature,
        "max_tokens": 2048
    }
    
    response = requests.post(OPENROUTER_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

# Quick test
test = generate_with_openrouter(
    "You are a helpful cooking assistant.",
    "In one sentence, what makes a good pasta sauce?"
)
print(f"Model: {OPENROUTER_MODEL}")
print(f"Response: {test}")

OpenRouter key loaded: Yes
Model: nvidia/nemotron-3-super-120b-a12b:free
Response: A good pasta sauce balances rich flavor, proper texture, and harmonious seasoning that clings to the pasta.


Updated above for model rotation since hitting rate limits even on 2 requests within 20-30 seconds

## MODEL ROTATION - OPENROUTER

In [ ]:
OPENROUTER_MODELS = [
    "nvidia/nemotron-3-super-120b-a12b:free",
    "nvidia/nemotron-3-nano-30b-a3b:free",
    "nvidia/llama-nemotron-embed-vl-1b-v2:free",
    "nvidia/nemotron-nano-12b-v2-vl:free", 
]

# Adding hugging face as fall back

Here adding hugging face incase all openrouter models are limited: Good question — let me clarify. In the proposed design, _try_openrouter is the generation. generate_with_openrouter does NOT make its own separate API call. Here's the entire flow:

generate_with_openrouter(system_prompt, user_message)
  │
  ├─ calls _try_openrouter(messages)    ← makes 1 API call (or up to 6 if rate limited)
  │    └─ returns the LLM's answer text (or None if all 6 fail)
  │
  ├─ if result exists → return it       ← DONE, no more API calls
  │
  └─ if None → calls _try_huggingface(messages)  ← fallback, 1-3 more API calls
       └─ returns the LLM's answer text (or None)

In [57]:
import time

# --- Provider 1: OpenRouter (primary) --- Try multiple free models in sequence to avoid rate limits
OPENROUTER_MODELS = [
    "nvidia/nemotron-3-super-120b-a12b:free", # works the best, largest model, so far no rate limit hits
    "nvidia/nemotron-nano-12b-v2-vl:free", # works better than the 30b model for some reason, maybe it's more fine-tuned on instructions?
    "nvidia/nemotron-3-nano-30b-a3b:free", # rank last
    #"nvidia/llama-nemotron-embed-vl-1b-v2:free", # error on this model
    
]

# --- Provider 2: HuggingFace Inference API (fallback) ---
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN", "").strip().strip('"')
HF_MODELS = [
   "MiniMaxAI/MiniMax-M2.7", # tested this one works and works well very large parameter model, rank this 1st. 
   "openai/gpt-oss-120b", # works well rank this one 2nd best after MiniMaxAI/MiniMax-M2.7
   "Qwen/Qwen2.5-7B-Instruct", # works well rank 3rd
   "meta-llama/Llama-3.1-8B-Instruct",  # works but not as good as MiniMaxAI/MiniMax-M2.7 for recipe questions, maybe because it's a smaller model (8B vs 13B) and less fine-tuned on instructions?
   # "mistralai/Mistral-7B-Instruct-v0.3", Error message this is not a chat model
   # "deepseek-ai/DeepSeek-R1", # this one outputs too much of its reasoning thinking leave as last it's also a massive model
   
]

print(f"OpenRouter key: {'Yes' if OPENROUTER_API_KEY else 'NO'}")
print(f"HuggingFace token: {'Yes' if HF_TOKEN else 'NO — HF fallback disabled'}")
print(f"Cascade: {len(OPENROUTER_MODELS)} OpenRouter models → {len(HF_MODELS)} HuggingFace models")

def _try_openrouter(messages, temperature):
    """Try each OpenRouter free model in sequence."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://discountmate.app",
        "X-Title": "DiscountMate RAG"
    }
    for model_name in OPENROUTER_MODELS:
        payload = {
            "model": model_name,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": 2048
        }
        response = requests.post(OPENROUTER_URL, headers=headers, json=payload)
        data = response.json()
        if response.status_code == 200:
            content = data.get("choices", [{}])[0].get("message", {}).get("content")
            if content:
                print(f"  [OpenRouter] Used model: {model_name}")
                return content
        elif response.status_code == 429:
            print(f"  [OpenRouter] {model_name} rate limited, trying next...")
        else:
            print(f"  [OpenRouter] {model_name} error {response.status_code}")
    return None


def _try_huggingface(messages, temperature):
    """Fallback: try HuggingFace Inference API models (OpenAI-compatible endpoint)."""
    if not HF_TOKEN:
        print("  [HuggingFace] No token — skipping fallback")
        return None
    
    for model_name in HF_MODELS:
        url = f"https://router.huggingface.co/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {HF_TOKEN}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": model_name,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": 2048
        }
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=60)
            # Guard against non-JSON responses (HTML error pages, empty body)
            try:
                data = response.json()
            except (ValueError, requests.exceptions.JSONDecodeError):
                print(f"  [HuggingFace] {model_name} returned non-JSON (status {response.status_code}), trying next...")
                continue
            
            if response.status_code == 200:
                content = data.get("choices", [{}])[0].get("message", {}).get("content")
                if content:
                    print(f"  [HuggingFace] Used model: {model_name}")
                    return content
            elif response.status_code == 503:
                print(f"  [HuggingFace] {model_name} loading (cold start), trying next...")
            elif response.status_code == 429:
                print(f"  [HuggingFace] {model_name} rate limited, trying next...")
            else:
                print(f"  [HuggingFace] {model_name} error {response.status_code}: {data.get('error', 'unknown')}")
        except requests.exceptions.Timeout:
            print(f"  [HuggingFace] {model_name} timed out, trying next...")
    return None



def generate_with_openrouter(system_prompt, user_message, temperature=0.7):
    """
    Cascading LLM generation: OpenRouter (6 models) → HuggingFace (3 models).
    Same function signature — all downstream code works without changes.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    # Tier 1: OpenRouter free models
    result = _try_openrouter(messages, temperature)
    if result:
        return result
    
    # Tier 2: HuggingFace Inference API fallback
    print("  All OpenRouter models exhausted — falling back to HuggingFace...")
    result = _try_huggingface(messages, temperature)
    if result:
        return result
    
    raise Exception(
        "All LLM providers exhausted (6 OpenRouter + 3 HuggingFace models). "
        "Try again in ~60s or check API keys in .env"
    )

OpenRouter key: Yes
HuggingFace token: Yes
Cascade: 3 OpenRouter models → 4 HuggingFace models


Original Generate with open router note used:

In [ ]:
#def generate_with_openrouter(system_prompt, user_message, temperature=0.7):
    """Try each free model in sequence until one responds."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://discountmate.app",
        "X-Title": "DiscountMate RAG"
    }
    
    for model_name in OPENROUTER_MODELS:
        payload = {
            "model": model_name,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            "temperature": temperature,
            "max_tokens": 2048
        }
        
        response = requests.post(OPENROUTER_URL, headers=headers, json=payload)
        data = response.json()
        
        if response.status_code == 200:
            print(f"  Used model: {model_name}")
            return data["choices"][0]["message"]["content"]
        elif response.status_code == 429:
            print(f"  {model_name} rate limited, trying next...")
            continue
        else:
            print(f"  {model_name} error {response.status_code}: {data.get('error', {}).get('message', 'unknown')}")
            continue
    
    raise Exception("All OpenRouter free models rate limited — try again in ~60s")

Full RAG pipeline: local search + OpenRouter generation

In [58]:
# Cell V2-9: Full V2 RAG Pipeline — sentence-transformers search + OpenRouter generation

SYSTEM_PROMPT = """You are DiscountMate's recipe assistant. You help users find and cook recipes 
from our recipe collection.

Rules:
- ONLY recommend recipes from the provided context — never invent recipes
- Include ALL ingredients with exact quantities
- Include full step-by-step cooking instructions
- Mention prep time, cook time, and servings
- If multiple recipes match, briefly describe each and let the user choose
- If no recipes match the query, say so honestly
- Be friendly and concise"""

def rag_query_v2(user_query, top_k=3):
    """Full RAG: local search + OpenRouter generation."""
    # Step 1: Retrieve relevant recipes
    results = search_recipes_v2(user_query, top_k=top_k)
    
    # Step 2: Build context from retrieved recipes
    context_parts = []
    for r in results:
        context_parts.append(f"[Recipe: {r['name']} | Score: {r['score']:.3f}]\n{r['text']}")
    context = "\n\n---\n\n".join(context_parts)
    
    # Step 3: Generate answer via OpenRouter
    user_message = f"""Based on these recipes from our database:

{context}

User question: {user_query}"""
    
    answer = generate_with_openrouter(SYSTEM_PROMPT, user_message)
    
    return {
        "answer": answer,
        "sources": [{"name": r["name"], "score": r["score"]} for r in results]
    }

# Test the full pipeline
print("=" * 60)
print("V2 RAG TEST: sentence-transformers + OpenRouter")
print("=" * 60)

result = rag_query_v2("I want to make something with chicken that's quick and easy")

print(f"\nSources:")
for s in result["sources"]:
    print(f"  {s['score']:.3f} | {s['name']}")

print(f"\n{'─' * 60}")
print(result["answer"])

V2 RAG TEST: sentence-transformers + OpenRouter
  [OpenRouter] Used model: nvidia/nemotron-3-super-120b-a12b:free

Sources:
  0.602 | Roast Chicken & Vegetables Recipe | Woolworths
  0.595 | Chicken Rolls FIlled with Apricot & Almonds Recipe | Woolworths
  0.580 | Chicken Pasta Soup with Mozzarella Toast Recipe | Woolworths

────────────────────────────────────────────────────────────
Here are the chicken recipes from our collection that are relatively quick and easy to put together:

| Recipe | Prep Time | Cook Time | Total | Servings | Why it’s quick/easy |
|--------|-----------|-----------|-------|----------|----------------------|
| **Chicken Rolls Filled with Apricot & Almonds** | 15 min | 30 min | ~45 min | 6 | Simple filling, pan‑sear then bake; only a few steps. |
| **Chicken Pasta Soup with Mozzarella Toast** | 10 min | 30 min | ~40 min | 4 | One‑pot soup plus quick toast; minimal hands‑on time. |
| Roast Chicken & Vegetables | 20 min | 70 min | ~90 min | 4 | Longer roast time

In [12]:
# Check if chicken recipes loaded in V2
chicken_recipes = [i for i, m in enumerate(recipe_metadata) if 'chicken' in m['name'].lower()]
print(f"Chicken recipes found: {len(chicken_recipes)}")
for i in chicken_recipes:
    print(f"  [{i}] {recipe_metadata[i]['name']}")
    
# If they exist, check their embedding norms
if chicken_recipes:
    idx = chicken_recipes[0]
    emb_norm = np.linalg.norm(recipe_embeddings[idx])
    print(f"\nEmbedding norm for first chicken recipe: {emb_norm:.4f}")
    print(f"Avg embedding norm across all recipes: {np.linalg.norm(recipe_embeddings, axis=1).mean():.4f}")

Chicken recipes found: 8
  [20] Chargrilled Chicken with Fresh Pineapple & Red Onion Recipe | Woolworths
  [25] Chicken & Mushroom Green Curry Recipe | Woolworths
  [26] Chicken, Leek & Mushroom Casserole Recipe | Woolworths
  [27] Chicken Pasta Soup with Mozzarella Toast Recipe | Woolworths
  [28] Chicken Rolls FIlled with Apricot & Almonds Recipe | Woolworths
  [43] Hearty Chicken Minestrone Soup Recipe | Woolworths
  [56] Moroccan Chicken with Couscous Salad Recipe | Woolworths
  [72] Roast Chicken & Vegetables Recipe | Woolworths

Embedding norm for first chicken recipe: 1.0000
Avg embedding norm across all recipes: 1.0000


## Phase 2 Implementation: PProduct matching demo — simulates on-sale products → ingredients → recipes

Here I'll try to go one step further by integratin 'on sale' products for a top-k=1 return from each retailer if on special. 

So for example if the ingredients list says "chopped onions" I'll try to add the best on sale item for choped onions per retailer so output will be something like "Chopped Onion [Chopped onions $2.99 coles on special etc]

- Simulated catalog has ~40 products across 3 retailers, with duplicates/variants (e.g. multiple chicken breast options at different prices/promo states)
- For each ingredient in the recipe, it embeds that ingredient string and finds the best cosine match from on-sale products only
- It keeps only the cheapest on-sale match per retailer (top-k=1 per retailer)
- If a retailer has no on-sale product scoring above the 0.35 threshold, it's simply omitted
- Output format shows the ingredient with bracketed product matches per retailer

In [20]:
# Cell V2-10b: Flow 2 — Recipe → Ingredient → Matching On-Sale Products
# Given a recipe result, find the best matching on-sale product per retailer per ingredient

# --- Simulated retailer product catalog ---
product_catalog = [
    # beef products
    {"retailer": "Woolworths", "name": "Beef Mince 500g", "price": 9.50, "on_sale": True},
    {"retailer": "Woolworths", "name": "Beef Steak 600g", "price": 8.00, "on_sale": False},
    {"retailer": "Coles", "name": "Coles RSPCA Beef Mince 500g", "price": 10.00, "on_sale": True},
    {"retailer": "Coles", "name": "Coles Beef Steak 500g", "price": 7.50, "on_sale": True},
    {"retailer": "IGA", "name": "IGA Beef Mince 450g", "price": 11.00, "on_sale": False},
    {"retailer": "IGA", "name": "Community Co Chicken Breast 500g", "price": 8.99, "on_sale": True},
    
    # Shallots
    {"retailer": "Woolworths", "name": "Brown Shallots 1kg Bag", "price": 2.50, "on_sale": True},
    {"retailer": "Woolworths", "name": "Red Shallots per kg", "price": 4.90, "on_sale": False},
    {"retailer": "Coles", "name": "Coles Brown Shallots 1kg", "price": 2.70, "on_sale": False},
    {"retailer": "Coles", "name": "Coles Diced Shallots 500g Frozen", "price": 3.00, "on_sale": True},
    {"retailer": "IGA", "name": "Brown Shallots Loose per kg", "price": 3.50, "on_sale": True},
    
    # Cabbage
    {"retailer": "Woolworths", "name": "Green Cabbage 1kg", "price": 2.50, "on_sale": True},
    {"retailer": "Woolworths", "name": "Red Cabbage 1kg", "price": 3.80, "on_sale": False},
    {"retailer": "Coles", "name": "Coles Green Cabbage 1kg", "price": 2.70, "on_sale": False},
    {"retailer": "Coles", "name": "Coles Red Cabbage 1kg", "price": 3.00, "on_sale": True},
    {"retailer": "IGA", "name": "Green Cabbage Loose per kg", "price": 3.50, "on_sale": True},
    
    # Flour
    {"retailer": "Woolworths", "name": "White Wings Self Raising Flour 1kg", "price": 2.20, "on_sale": True},
    {"retailer": "Coles", "name": "Coles Self Raising Flour 1kg", "price": 1.80, "on_sale": True},
    {"retailer": "IGA", "name": "Defiance Self Raising Flour 1kg", "price": 2.50, "on_sale": False},
    
    # Garlic
    {"retailer": "Woolworths", "name": "Fresh Garlic Bulb 3 Pack", "price": 3.00, "on_sale": False},
    {"retailer": "Coles", "name": "Coles Crushed Garlic 240g", "price": 2.80, "on_sale": True},
    {"retailer": "IGA", "name": "Fresh Garlic Bulbs 3pk", "price": 3.50, "on_sale": True},
    
    # Olive oil
    {"retailer": "Woolworths", "name": "Woolworths Extra Virgin Olive Oil 500mL", "price": 7.00, "on_sale": True},
    {"retailer": "Coles", "name": "Coles Extra Virgin Olive Oil 500mL", "price": 6.50, "on_sale": False},
    {"retailer": "IGA", "name": "Cobram Estate Olive Oil 375mL", "price": 8.00, "on_sale": True},
    
    # Mushrooms
    {"retailer": "Woolworths", "name": "Cup Mushrooms 200g Punnet", "price": 3.50, "on_sale": True},
    {"retailer": "Coles", "name": "Coles Sliced Mushrooms 200g", "price": 3.00, "on_sale": True},
    {"retailer": "IGA", "name": "Button Mushrooms 200g", "price": 4.00, "on_sale": False},
    
    # Coconut milk/cream
    {"retailer": "Woolworths", "name": "Woolworths Coconut Cream 400mL", "price": 1.50, "on_sale": True},
    {"retailer": "Coles", "name": "Coles Coconut Milk 400mL", "price": 1.40, "on_sale": True},
    {"retailer": "IGA", "name": "Ayam Coconut Cream 270mL", "price": 2.80, "on_sale": False},
    
    # Soy sauce
    {"retailer": "Woolworths", "name": "Kikkoman Soy Sauce 250mL", "price": 3.50, "on_sale": False},
    {"retailer": "Coles", "name": "Coles Soy Sauce 500mL", "price": 2.00, "on_sale": True},
    {"retailer": "IGA", "name": "Kikkoman Soy Sauce 150mL", "price": 3.00, "on_sale": True},
    
    # Salmon
    {"retailer": "Woolworths", "name": "Tasmanian Salmon Fillets 2pk 280g", "price": 14.00, "on_sale": True},
    {"retailer": "Coles", "name": "Coles Salmon Portions 2pk 250g", "price": 12.50, "on_sale": True},
    {"retailer": "IGA", "name": "Atlantic Salmon Fillet per kg", "price": 32.00, "on_sale": False},
    
    # oil
    {"retailer": "Woolworths", "name": "San oil Spaghetti 500m", "price": 1.95, "on_sale": True},
    {"retailer": "Coles", "name": "Coles virgin oil 500mL", "price": 1.00, "on_sale": True},
    {"retailer": "IGA", "name": "yarra oil 1L", "price": 2.50, "on_sale": True},
]

# --- Embed all product names ---
print("Embedding product catalog...")
product_names = [p["name"] for p in product_catalog]
product_embeddings = model.encode(product_names, show_progress_bar=True, batch_size=32)
print(f"Embedded {len(product_embeddings)} products ({product_embeddings.shape})")


# --- Flow 2: Annotate recipe ingredients with matching on-sale products ---

def match_products_to_ingredient(ingredient_text, threshold=0.55):
    """
    For a single ingredient, find the top-1 cheapest on-sale product per retailer.
    Returns dict: {retailer: {name, price}} or empty if no match above threshold.
    """
    # Embed the ingredient
    ing_emb = model.encode([ingredient_text])
    
    # Cosine similarity against all products
    ing_norm = ing_emb / (np.linalg.norm(ing_emb, axis=1, keepdims=True) + 1e-10)
    prod_norms = product_embeddings / (np.linalg.norm(product_embeddings, axis=1, keepdims=True) + 1e-10)
    sims = np.dot(prod_norms, ing_norm.T).flatten()
    
    # Filter: on_sale only, above threshold
    matches_by_retailer = {}
    for idx in np.argsort(sims)[::-1]:
        score = float(sims[idx])
        if score < threshold:
            break
        product = product_catalog[idx]
        if not product["on_sale"]:
            continue
        retailer = product["retailer"]
        # Keep only the cheapest per retailer
        if retailer not in matches_by_retailer:
            matches_by_retailer[retailer] = {
                "name": product["name"],
                "price": product["price"],
                "score": score
            }
        elif product["price"] < matches_by_retailer[retailer]["price"]:
            matches_by_retailer[retailer] = {
                "name": product["name"],
                "price": product["price"],
                "score": score
            }
    
    return matches_by_retailer


def annotate_recipe_ingredients(recipe_idx):
    """Take a recipe and annotate each ingredient with matching on-sale products."""
    meta = recipe_metadata[recipe_idx]
    print(f"\n{'='*70}")
    print(f"📖 Recipe: {meta['name']}")
    print(f"   Cuisine: {meta['cuisine']} | Prep: {meta['prep_time']} | Cook: {meta['cook_time']}")
    print(f"{'='*70}")
    print(f"\nIngredients with matching on-sale products:\n")
    
    for ing in meta["ingredients"]:
        matches = match_products_to_ingredient(ing)
        print(f"  • {ing}")
        if matches:
            for retailer in ["Woolworths", "Coles", "IGA"]:
                if retailer in matches:
                    m = matches[retailer]
                    print(f"      [{retailer} - {m['name']} ${m['price']:.2f}] (match: {m['score']:.2f})")
        else:
            print(f"      [No matching products on sale]")
        print()


# --- TEST: Pick a recipe and annotate it ---
# Use search to find a recipe first
results = search_recipes_v2("chicken stir fry", top_k=1)
if results:
    # Find this recipe's index in recipe_metadata
    recipe_name = results[0]["name"]
    recipe_idx = next(i for i, m in enumerate(recipe_metadata) if m["name"] == recipe_name)
    annotate_recipe_ingredients(recipe_idx)
else:
    # Fallback: just use the first chicken recipe
    chicken_idx = next(i for i, m in enumerate(recipe_metadata) if 'chicken' in m['name'].lower())
    annotate_recipe_ingredients(chicken_idx)

Embedding product catalog...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:00<00:00,  6.90it/s]


Embedded 40 products ((40, 768))

📖 Recipe: Beef & Vegetable Stir-fry Recipe | Woolworths
   Cuisine: Chinese | Prep: 15m | Cook: 10m

Ingredients with matching on-sale products:

  • 350g cabbage, finely shredded
      [Woolworths - Green Cabbage 1kg $2.50] (match: 0.83)
      [Coles - Coles Red Cabbage 1kg $3.00] (match: 0.78)
      [IGA - Green Cabbage Loose per kg $3.50] (match: 0.72)

  • 1 large carrot, cut into matchsticks
      [Woolworths - Green Cabbage 1kg $2.50] (match: 0.58)

  • 5 green shallots, thinley sliced
      [Woolworths - Brown Shallots 1kg Bag $2.50] (match: 0.70)
      [Coles - Coles Diced Shallots 500g Frozen $3.00] (match: 0.69)
      [IGA - Brown Shallots Loose per kg $3.50] (match: 0.59)

  • 0.5 cup Hoi Sin sauce
      [Coles - Coles Soy Sauce 500mL $2.00] (match: 0.56)

  • 1 tbs sesame seeds
      [No matching products on sale]

  • 500g beef strips
      [Woolworths - Beef Mince 500g $9.50] (match: 0.75)
      [Coles - Coles Beef Steak 500g $7.50] (matc

## Phase 2: Full RAG with ingredient 'On Special' matching

In [23]:
# Cell V2-9b: Full RAG with product annotations

def build_annotated_recipe(recipe_idx):
    """Build recipe text with on-sale product matches inline."""
    meta = recipe_metadata[recipe_idx]
    
    # Build annotated ingredient list
    annotated_ingredients = []
    for ing in meta["ingredients"]:
        matches = match_products_to_ingredient(ing, threshold=0.65)
        if matches:
            # Format: ingredient [Retailer - Product $price]
            annotations = []
            for retailer in ["Woolworths", "Coles", "IGA"]:
                if retailer in matches:
                    m = matches[retailer]
                    annotations.append(f"{retailer} - {m['name']} ${m['price']:.2f}")
            annotated_ingredients.append(f"- {ing} [{' | '.join(annotations)}]")
        else:
            annotated_ingredients.append(f"- {ing}")
    
    # Rebuild recipe text with annotated ingredients
    text = f"""Recipe: {meta['name']}
Description: {meta['description']}
Category: {meta['category']} | Cuisine: {meta['cuisine']}
Prep: {meta['prep_time']} | Cook: {meta['cook_time']} | Serves: {meta['servings']}

Ingredients (with matching on-sale products):
{chr(10).join(annotated_ingredients)}

Instructions:
{chr(10).join(f'{j+1}. {s}' for j, s in enumerate(meta['instructions']))}"""
    
    return text


def rag_query_v2_with_products(user_query, top_k=3):
    """Full RAG with inline product matching."""
    # Step 1: Retrieve relevant recipes
    results = search_recipes_v2(user_query, top_k=top_k)
    
    # Step 2: Build annotated context (ingredients get product matches)
    context_parts = []
    for r in results:
        recipe_idx = next(i for i, m in enumerate(recipe_metadata) if m["name"] == r["name"])
        annotated_text = build_annotated_recipe(recipe_idx)
        context_parts.append(f"[Recipe: {r['name']} | Score: {r['score']:.3f}]\n{annotated_text}")
    context = "\n\n---\n\n".join(context_parts)
    
    # Step 3: Generate — LLM sees the product annotations naturally
    system_prompt = """You are DiscountMate's recipe assistant. You help users find and cook recipes.

Rules:
- ONLY recommend recipes from the provided context
- Include ALL ingredients with exact quantities
- Where products are shown in [brackets] after an ingredient, include them as shopping suggestions
- Format product matches clearly so users can see which retailers have deals
- Include full step-by-step cooking instructions
- Be friendly and concise"""
    
    user_message = f"""Based on these recipes from our database:

{context}

User question: {user_query}"""
    
    answer = generate_with_openrouter(system_prompt, user_message)
    
    return {
        "answer": answer,
        "sources": [{"name": r["name"], "score": r["score"]} for r in results]
    }

# Test it
print("=" * 60)
print("V2 RAG + PRODUCT MATCHING TEST")
print("=" * 60)
result = rag_query_v2_with_products("give me two beef dishes")
print(f"\nSources:")
for s in result["sources"]:
    print(f"  {s['score']:.3f} | {s['name']}")
print(f"\n{'─' * 60}")
print(result["answer"])

V2 RAG + PRODUCT MATCHING TEST
  google/gemma-4-26b-a4b-it:free rate limited, trying next...
  google/gemma-4-31b-it:free rate limited, trying next...
  Used model: nvidia/nemotron-3-super-120b-a12b:free

Sources:
  0.558 | Beef & Vegetable Casserole Recipe | Woolworths
  0.525 | Rich Beef Casserole Recipe | Woolworths
  0.451 | Pot Roast with Carrots, Mushrooms & Mash Recipe | Woolworths

────────────────────────────────────────────────────────────
Here are two hearty beef dishes you can make right now, pulled straight from the Woolworths‑sourced recipes in our database. Each lists every ingredient (with exact amounts) and highlights any on‑sale product matches so you know where to shop for the best price.

---

## 1️⃣ Beef & Vegetable Casserole  
*Source: Woolworths – Score 0.558*  
**Category:** Mains • **Cuisine:** French  
**Prep:** 20 min • **Cook:** 2 h • **Serves:** 4  

### Ingredients  
| Ingredient | Quantity | On‑sale product suggestions |
|------------|----------|---------

## phase 3: 

In [ ]:
# Cell V2-9c: Multi-turn conversation RAG (for future backend/chatbox integration)
# No LangChain needed — just maintain a messages list across calls.
# Each user/assistant turn is appended so the LLM has full conversation context.

conversation_history = []

CHAT_SYSTEM_PROMPT = """You are DiscountMate's recipe assistant. You help users find and cook recipes.

Rules:
- ONLY recommend recipes from the provided context — never invent recipes
- Include ALL ingredients with exact quantities
- Where products are shown in [brackets] after an ingredient, include them as shopping suggestions
- Format product matches clearly so users can see which retailers have deals
- Include full step-by-step cooking instructions
- If the user asks a follow-up about a recipe already discussed, refer back to it
- Be friendly and concise
- Do NOT say "let me know if you'd like more" — the user can always ask follow-ups"""

def rag_chat(user_query, top_k=3):
    """
    Multi-turn RAG chat — maintains conversation history across calls.
    First message retrieves recipes + annotates with products.
    Follow-up messages reuse the existing context in history.
    """
    # Retrieve + annotate recipes
    results = search_recipes_v2(user_query, top_k=top_k)
    context_parts = []
    for r in results:
        recipe_idx = next(i for i, m in enumerate(recipe_metadata) if m["name"] == r["name"])
        context_parts.append(build_annotated_recipe(recipe_idx))
    context = "\n\n---\n\n".join(context_parts)
    
    # First message includes recipe context; follow-ups just add user text
    if not conversation_history:
        user_msg = f"Recipes from our database:\n\n{context}\n\nUser: {user_query}"
    else:
        # Follow-up — add fresh context if the query seems like a new topic,
        # otherwise just pass the user's message (LLM has prior context in history)
        user_msg = f"[Additional recipes if needed:\n{context}]\n\nUser: {user_query}"
    
    # Append user turn
    conversation_history.append({"role": "user", "content": user_msg})
    
    # Build full message payload with history
    messages = [{"role": "system", "content": CHAT_SYSTEM_PROMPT}] + conversation_history
    
    # Send to OpenRouter with model rotation
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://discountmate.app",
        "X-Title": "DiscountMate RAG Chat"
    }
    
    answer = None
    for model_name in OPENROUTER_MODELS:
        payload = {
            "model": model_name,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 2048
        }
        response = requests.post(OPENROUTER_URL, headers=headers, json=payload)
        data = response.json()
        if response.status_code == 200:
            print(f"  Used model: {model_name}")
            answer = data["choices"][0]["message"]["content"]
            break
        elif response.status_code == 429:
            print(f"  {model_name} rate limited, trying next...")
            continue
    
    if answer is None:
        answer = "All models rate limited — try again in ~60s"
    
    # Append assistant turn — this is what gives multi-turn memory
    conversation_history.append({"role": "assistant", "content": answer})
    
    return {
        "answer": answer,
        "sources": [{"name": r["name"], "score": r["score"]} for r in results],
        "turns": len(conversation_history) // 2
    }

def reset_chat():
    """Clear conversation history to start fresh."""
    conversation_history.clear()
    print("Chat history cleared — starting fresh conversation.")

# --- DEMO: Multi-turn conversation ---
print("=" * 60)
print("MULTI-TURN CHAT DEMO")
print("=" * 60)

reset_chat()

# Turn 1
print("\n USER: What beef recipes do you have?")
r1 = rag_chat("What beef recipes do you have?")
print(f" MATEBOT:\n{r1['answer']}")
print(f"\n(Turn {r1['turns']} | Sources: {', '.join(s['name'] for s in r1['sources'])})")

print(f"\n{'─'*60}")

# Turn 2 — follow-up referencing previous answer
print("\n USER: Show me the full recipe for the first one")
r2 = rag_chat("Show me the full recipe for the first one")
print(f"🤖 MATEBOT:\n{r2['answer']}")
print(f"\n(Turn {r2['turns']})")

MULTI-TURN CHAT DEMO
Chat history cleared — starting fresh conversation.

👤 USER: What beef recipes do you have?
  google/gemma-4-26b-a4b-it:free rate limited, trying next...
  google/gemma-4-31b-it:free rate limited, trying next...
  Used model: nvidia/nemotron-3-super-120b-a12b:free
🤖 MATEBOT:
Here are the beef recipes currently in our database:

---

### 1. Beef & Vegetable Casserole  
**Source:** Woolworths  
**Category:** Mains | **Cuisine:** French  
**Prep:** 20 min | **Cook:** 120 min | **Serves:** 4  

**Ingredients**  
- 0.25 cup olive oil [IGA - Cobram Estate Olive Oil 375 mL $8.00]  
- 1.2 kg chuck steak [Coles - Coles Beef Steak 500 g $7.50]  
- 0.25 cup plain flour  
- 1 onion, diced  
- 2 parsnips, peeled, core removed, diced  
- 2 potatoes, peeled, diced  
- 2 carrots, peeled, diced  
- 2 stick celery, sliced  
- 800 g diced tomatoes  
- 1 cup beef stock  
- 0.5 cup flat‑leaf parsley, roughly chopped  

**Instructions**  
1. Heat 1 Tbsp oil in a large heatproof casserol

Not used this was the initial V2 test but it looked up ingredients and matched to recipes, we just need to switch the lookup to return the on special products to the returned recipes first. 

In [71]:
# Cell V2-10: Test ingredient-based search (future product matching preview)
# This shows how we'll match on-sale products → ingredients → recipes

def search_ingredients(product_name, top_k=5):
    """Find ingredients most similar to a product name."""
    query_emb = model.encode([product_name])
    sims = np.dot(ingredient_embeddings, query_emb.T).flatten()
    top_indices = np.argsort(sims)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        ing = all_ingredients[idx]
        recipe_indices = ingredient_to_recipes[ing]
        recipe_names = [recipe_metadata[i]["name"] for i in recipe_indices]
        results.append({
            "ingredient": ing,
            "score": float(sims[idx]),
            "recipes": recipe_names
        })
    return results

# Simulate: "Woolworths Chicken Breast 500g" is on sale
print("PRODUCT MATCHING DEMO")
print("=" * 60)
test_products = [
    "chicken breast fillets",
    "self-raising flour",
    "arborio rice",
    "fresh salmon fillet"
]

for product in test_products:
    print(f"\n🏷️  On sale: '{product}'")
    matches = search_ingredients(product, top_k=3)
    for m in matches:
        print(f"  {m['score']:.3f} | {m['ingredient']}")
        print(f"         → Recipes: {', '.join(m['recipes'][:2])}")
    print()

PRODUCT MATCHING DEMO

🏷️  On sale: 'chicken breast fillets'
  0.806 | 2 chicken breast fillets
         → Recipes: Moroccan Chicken with Couscous Salad Recipe | Woolworths
  0.805 | 4 chicken breast fillets
         → Recipes: Chargrilled Chicken with Fresh Pineapple & Red Onion Recipe | Woolworths
  0.768 | 750g chicken breast fillets
         → Recipes: Chicken Rolls FIlled with Apricot & Almonds Recipe | Woolworths


🏷️  On sale: 'self-raising flour'
  0.796 | 2 cup self-raising flour
         → Recipes: Apple & Walnut Coffee Cake Recipe | Woolworths, Blueberry Streusel Cake Recipe | Woolworths
  0.765 | 2 cup self raising flour
         → Recipes: Mango Butter Cakes Recipe | Woolworths
  0.541 | 2 tbs flour
         → Recipes: Chicken Rolls FIlled with Apricot & Almonds Recipe | Woolworths


🏷️  On sale: 'arborio rice'
  0.811 | 2 cup arborio rice
         → Recipes: Pumpkin Risotto Recipe | Woolworths
  0.767 | 0.5 cup arborio rice
         → Recipes: Squid Stuffed with Spanish R

### 	Save all indexes as .npz files for GCP deployment

In [ ]:
# Cell V2-11: Save all embeddings and indexes as .npz files
# These can be loaded later for the full backend or GCP deployment

SAVE_DIR = "./v2_indexes"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Recipe embeddings + metadata
np.savez_compressed(
    os.path.join(SAVE_DIR, "recipe_index.npz"),
    embeddings=recipe_embeddings,
    texts=np.array(recipe_texts, dtype=object),
    names=np.array([m["name"] for m in recipe_metadata], dtype=object),
    categories=np.array([m["category"] for m in recipe_metadata], dtype=object),
    cuisines=np.array([m["cuisine"] for m in recipe_metadata], dtype=object),
    source_files=np.array([m["source_file"] for m in recipe_metadata], dtype=object)
)

# 2. Ingredient embeddings + mapping
# Save mapping as JSON (numpy can't store dicts of lists)
import json
np.savez_compressed(
    os.path.join(SAVE_DIR, "ingredient_index.npz"),
    embeddings=ingredient_embeddings,
    ingredients=np.array(all_ingredients, dtype=object)
)
with open(os.path.join(SAVE_DIR, "ingredient_to_recipes.json"), 'w') as f:
    json.dump(ingredient_to_recipes, f)

# 3. Full metadata as JSON (for the generation step)
with open(os.path.join(SAVE_DIR, "recipe_metadata.json"), 'w') as f:
    json.dump(recipe_metadata, f, indent=2)

# Report sizes
for fname in os.listdir(SAVE_DIR):
    fpath = os.path.join(SAVE_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname}: {size_kb:.1f} KB")

print(f"\nAll indexes saved to {SAVE_DIR}/")
print("These files are all you need for the production backend.")

### Verify saved indexes load correctly (cold start test)

In [ ]:
# Cell V2-12: Verify saved indexes load correctly (simulates production cold start)

# Load recipe index
recipe_data = np.load(os.path.join(SAVE_DIR, "recipe_index.npz"), allow_pickle=True)
loaded_recipe_embs = recipe_data["embeddings"]
loaded_recipe_names = recipe_data["names"]
print(f"Recipe index: {loaded_recipe_embs.shape[0]} recipes, {loaded_recipe_embs.shape[1]} dims")

# Load ingredient index
ing_data = np.load(os.path.join(SAVE_DIR, "ingredient_index.npz"), allow_pickle=True)
loaded_ing_embs = ing_data["embeddings"]
loaded_ings = ing_data["ingredients"]
print(f"Ingredient index: {loaded_ing_embs.shape[0]} ingredients, {loaded_ing_embs.shape[1]} dims")

# Load mapping
with open(os.path.join(SAVE_DIR, "ingredient_to_recipes.json"), 'r') as f:
    loaded_mapping = json.load(f)
print(f"Ingredient→recipe mapping: {len(loaded_mapping)} entries")

# Quick search test using loaded data
query_emb = model.encode(["vegetarian pasta"])
sims = np.dot(loaded_recipe_embs, query_emb.T).flatten()
top_idx = np.argsort(sims)[::-1][0]
print(f"\nTest search 'vegetarian pasta' → {loaded_recipe_names[top_idx]} ({sims[top_idx]:.3f})")
print("\n✅ All indexes verified — ready for production deployment")

## V2 Summary & Architecture Notes

### What we built
| Component | V1 (Google AI) | V2 (Local + OpenRouter) |
|---|---|---|
| Embeddings | `gemini-embedding-001` (API, rate limited) | `all-mpnet-base-v2` (local, unlimited) |
| Generation | `gemini-2.5-flash-lite` (API, daily quota) | `gemini-2.0-flash-exp:free` via OpenRouter |
| Ingredient index | ❌ Not extracted | ✅ Separate vector store |
| Storage | In-memory only | `.npz` files (GCP-ready) |
| 13k recipe processing | ~3-4 hours (rate limits) | ~2-3 minutes (local) |

### Production deployment path
1. Run this notebook on all 13k recipes → save `.npz` files
2. Copy `.npz` files to `Backend/ml-service/recipe_rag/indexes/`
3. Update `rag_pipeline.py` to load `.npz` + use OpenRouter for generation
4. Ingredient→product matching: embed product catalog with same model, cosine join to ingredient index

### Key insight
Same embedding model = same vector space. Products embedded with `all-mpnet-base-v2` are 
directly comparable to ingredients embedded with `all-mpnet-base-v2`. No training needed.